# rewriter 모듈 순차 실행 확인

`backend/rag/query_analyzer/` 안의 `.py` 모듈을 **import 해서 함수만 호출**합니다 (새 로직 없음).
모듈은 노트북 `user_question_rewrite_v3.ipynb` 를 `backend/rag/query_analyzer/` 와 같은 파일 구성으로 나눈 것입니다. 사전은 `backend/rag/query_analyzer/dictionary/term_dict.json`, `rule_dict.json` 을 읽습니다.

| 단계 | 모듈 | 호출 함수 |
|:---:|---|---|
| 0 | `config.py` | `settings()`, `check_hcx()` |
| 2-1 | `terms.py` | `lookup_terms()`, `build_term_block()` |
| 2-2 | `extract.py` | `rule_extract()` |
| 2-3 | `slots.py` | `build_slot_messages()`, `call_slots()` ← **HCX 1회** |
| 2-4 | `validate.py` | `validate_slots()`, `show_validated()` |
| 3 | `sentence.py` | `build_sentences()` |
| 4 | `bundle.py` | `make_bundle()`, `load_bundles()` |
| 5 | `bundle.py` · `trace.py` | `run_pipeline()`, `bundles_table()`, `build_trace()` |

HCX 호출: 0-3 에서 1회, 2-3 에서 1회, 5 에서 `N_CSV` 회. 결과 파일은 **저장하지 않습니다** (저장 셀은 주석 처리).

## 0. 모듈 불러오기 · 환경 확인 (`config.py`)

In [1]:
import sys
from datetime import date
from pathlib import Path

# 모듈 위치: <프로젝트 루트>/backend/rag/query_analyzer  (user_question_rewrite_v3.ipynb 를 나눈 것)
# 노트북(notebooks/query_rewrite/)에서 위로 올라가며 프로젝트 루트를 찾아 import 경로에 추가
# 프로젝트 루트가 경로에 있어야 모듈 안의 backend.core.llm (HCX 호출) 도 불러올 수 있음
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend" / "rag" / "query_analyzer").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from backend.rag.query_analyzer import config, terms, extract, slots, validate, sentence, bundle, trace

print("모듈 위치:", Path(config.__file__).parent)
config.settings()          # 키 값은 보여 주지 않음

모듈 위치: c:\dev\t4core-stat-agent\MJ\query_rewrite\query_analyzer


{'env_path': 'C:\\dev\\t4core-stat-agent\\.env',
 'hcx_host': 'https://clovastudio.stream.ntruss.com',
 'model': 'HCX-007',
 'app_type': 'testapp',
 'has_api_key': True,
 'term_dict': 'C:\\dev\\t4core-stat-agent\\MJ\\query_rewrite\\dictionary\\term_dict.json',
 'rule_dict': 'C:\\dev\\t4core-stat-agent\\MJ\\query_rewrite\\dictionary\\rule_dict.json',
 'output_dir': 'C:\\dev\\t4core-stat-agent\\MJ\\query_rewrite\\outputs'}

### 0-3. HCX 호출 확인 (1회)

In [2]:
config.check_hcx()

{'ok': True, 'elapsed_s': 0.76, 'response': '확인'}

## 1. 사용자 질문 입력

In [3]:
QUESTION = "2025년 전국에서 혼자 사는 가구는 몇 가구로 집계됐지?"
TODAY = date(2026, 11, 2)      # '작년', '최근 5년' 같은 상대 시점의 기준일 (None 이면 오늘)
QUESTION

'2025년 전국에서 혼자 사는 가구는 몇 가구로 집계됐지?'

## 2-2. 코드 추출 (`extract.rule_extract`)
상대 시점을 바꾼 질문, 코드가 찾은 값, 변환 내역을 돌려줍니다.

In [4]:
q2, rule_found, time_notes = extract.rule_extract(QUESTION, TODAY)
print("변환된 질문:", q2)
print("변환 내역  :", time_notes)
rule_found

변환된 질문: 2025년 전국에서 혼자 사는 가구는 몇 가구로 집계됐지?
변환 내역  : []


{'시점': ['2025년'], '지역': ['전국']}

## 2-1. 용어 사전 (`terms.lookup_terms`)
`slots.build_slot_messages` 와 같은 순서로, 2-2 에서 바꾼 질문에 사전을 적용합니다.

In [5]:
term_hits = terms.lookup_terms(q2)
print(terms.build_term_block(term_hits) or "(사전에서 찾은 표현 없음)")
term_hits

[용어 사전] 질문의 일상어는 아래 통계 용어로 바꾼다.
- 혼자 사는 가구 → 1인가구


[('혼자 사는 가구', ['1인가구'])]

## 2-3. HCX 슬롯 추출 (`slots`)
### 프롬프트 확인 (HCX 호출 없음)

In [6]:
messages, info = slots.build_slot_messages(QUESTION, TODAY)
print(messages[1]["content"])          # user 프롬프트 (system 프롬프트는 messages[0])

[오늘 날짜] 2026-11-02

[용어 사전] 질문의 일상어는 아래 통계 용어로 바꾼다.
- 혼자 사는 가구 → 1인가구

[코드가 질문에서 찾은 값] 맞는 슬롯에 그대로 넣는다.
- 시점: 2025년
- 지역: 전국

<question>2025년 전국에서 혼자 사는 가구는 몇 가구로 집계됐지?</question>


### HCX 호출 (1회)

In [7]:
r = slots.call_slots(QUESTION, TODAY)
print("성공:", r["parsed"], "|", r["elapsed_s"], "초", "|", r["error"] or "")
r["hcx_out"]

성공: True | 2.01 초 | 


{'slots': {'시점': ['2025년'],
  '지역': ['전국'],
  '전출지': [],
  '전입지': [],
  '국적': [],
  '대상': ['1인가구'],
  '성별': [],
  '연령': [],
  '조건': [],
  '이동': [],
  '지표': ['가구수'],
  '비교축': [],
  '의도': ['값 조회']},
 'sub_questions': []}

## 2-4. 슬롯 검증 (`validate.validate_slots`)

In [8]:
v = validate.validate_slots(r)
validate.show_validated(r, v)

원 질문 : 2025년 전국에서 혼자 사는 가구는 몇 가구로 집계됐지?
HCX 슬롯: {'시점': ['2025년'], '지역': ['전국'], '대상': ['1인가구'], '지표': ['가구수'], '의도': ['값 조회']}
고친 내역:
  - 없음
검증 슬롯: {'시점': ['2025년'], '지역': ['전국'], '대상': ['1인가구'], '지표': ['가구수'], '의도': ['값 조회']}


## 3. 슬롯 → 문장 (`sentence.build_sentences`)

In [9]:
sres = sentence.build_sentences(v)
sres

{'rewritten': '2025년 전국 1인가구 가구수', 'sub_questions': []}

## 4. QueryBundle (`bundle.make_bundle`)

In [10]:
b = bundle.make_bundle(QUESTION, v, sres)
b

QueryBundle(original='2025년 전국에서 혼자 사는 가구는 몇 가구로 집계됐지?', rewritten='2025년 전국 1인가구 가구수', sub_questions=[], intent=['값 조회'], filters={'시점': ['2025년'], '지역': ['전국'], '대상': ['1인가구'], '지표': ['가구수']}, sub_filters=None)

### 저장된 결과와 비교 (`bundle.load_bundles`)
`outputs/query_bundles_*.json` 중 가장 최근 실행 결과에 같은 질문이 있으면 나란히 봅니다.

In [11]:
saved = {x.original: x for x in bundle.load_bundles()}
print("지금  :", b.rewritten)
print("저장본:", saved[QUESTION].rewritten if QUESTION in saved else "(저장본에 없는 질문)")

지금  : 2025년 전국 1인가구 가구수
저장본: 2025년 전국 1인가구 가구수


## 5. 평가 질문 여러 개 한 번에 (`bundle.run_pipeline`, `trace.build_trace`)
`run_pipeline` 은 2-3 → 2-4 → 3 → 4 를 한 번에 실행합니다. `storage/test_data/kosis_rag_eval_80_queries.csv` (`config.DATA_DIR`) 의 `question` 열만 씁니다.

In [12]:
import pandas as pd

N_CSV = 3                                                  # HCX 를 N_CSV 회 호출
questions = pd.read_csv(config.DATA_DIR / "kosis_rag_eval_80_queries.csv",
                        encoding="utf-8-sig", usecols=["question"])["question"].astype(str).str.strip().tolist()[:N_CSV]
results = [bundle.run_pipeline(q, TODAY) for q in questions]
bundle.bundles_table([res["bundle"] for res in results])

,original,rewritten,sub_questions,intent,filters,sub_filters
0,2025년 전국에서 혼자 사는 가구는 몇 가구로 집계됐지?,2025년 전국 1인가구 가구수,[],[값 조회],"{'시점': ['2025년'], '지역': ['전국'], '대상': ['1인가구']...",None
1,2024년에서 2025년 사이 다문화가구는 전국적으로 몇 가구 늘었나?,2024~2025년 전국 다문화가구 가구수 증감,[],[증감],"{'시점': ['2024년', '2025년'], '지역': ['전국'], '대상':...",None
2,2016년과 2025년의 전국 한부모가구 차이를 가구 수로 말해 줘.,"2016년, 2025년 전국 한부모 가구수 증감",[],[증감],"{'시점': ['2016년', '2025년'], '지역': ['전국'], '대상':...",None


In [13]:
tr = trace.build_trace(results, TODAY, source="kosis_rag_eval_80_queries.csv")   # 대시보드용 중간 결과 (메모리에만)
print({k: v for k, v in tr["meta"].items() if k != "system_prompt"})   # 시스템 프롬프트는 길어서 뺌
tr["items"][0].keys()

{'created': '2026-11-05T15:50:39', 'model': 'HCX-007', 'today': '2026-11-02', 'source': 'kosis_rag_eval_80_queries.csv', 'n': 3, 'slot_keys': ['시점', '지역', '전출지', '전입지', '국적', '대상', '성별', '연령', '조건', '이동', '지표', '비교축']}


dict_keys(['no', 'question', 'term_hits', 'input_question', 'time_notes', 'rule_found', 'user_prompt', 'hcx_out', 'hcx_slots', 'hcx_subs', 'parsed', 'error', 'elapsed_s', 'valid_slots', 'valid_subs', 'valid_log', 'rewritten', 'sub_questions', 'bundle'])

In [14]:
# 파일로 저장하려면 주석 해제 (outputs/query_bundles_YYYYMMDD_HHMMSS.json 새 파일, dashboard/rewrite_trace.* 는 덮어씀)
# bundle.save_bundles([res["bundle"] for res in results])
# trace.save_trace(tr)